In [6]:
#!/usr/bin/env python3

# This file is part of the Interface Reconstruction Library (IRL),
# a library for interface reconstruction and computational geometry operations.
#
# Copyright (C) 2026 Andrew Cahaly <ajc428@cornell.edu>
#
# This Source Code Form is subject to the terms of the Mozilla Public
# License, v. 2.0. If a copy of the MPL was not distributed with this
# file, You can obtain one at https://mozilla.org/MPL/2.0/.

# Generates the plicnet.h file for a given Pytorch model
import sys
!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install torch
import torch
import numpy as np
import os

In [7]:
torch.set_default_dtype(torch.float64)
# Use a very large linewidth so NumPy formats the inner arrays naturally
np.set_printoptions(threshold=np.inf)
np.set_printoptions(linewidth=1900)
model = torch.jit.load('./model.pt')

In [8]:

def format_array_content(content, max_len=1900):
    content = content.strip()
    if len(content) <= max_len:
        return content
    lines = []
    while len(content) > 0:
        if len(content) <= max_len:
            lines.append(content)
            break
        break_pos = content.rfind(',', 0, max_len)
        if break_pos == -1:
            break_pos = max_len
        lines.append(content[:break_pos+1])
        content = "    " + content[break_pos+1:].lstrip()
    return "\n".join(lines)

In [19]:
file = open("../plicnet_weights_and_biases.h", "w")

print("// This file is part of the Interface Reconstruction Library (IRL),", file=file)
print("// a library for interface reconstruction and computational geometry operations.", file=file)
print("//", file=file)
print("// Copyright (C) 2026 Andrew Cahaly <ajc428@cornell.edu>", file=file)
print("//", file=file)
print("// This Source Code Form is subject to the terms of the Mozilla Public", file=file)
print("// License, v. 2.0. If a copy of the MPL was not distributed with this", file=file)
print("// file, You can obtain one at https://mozilla.org/MPL/2.0/.\n", file=file)
print("// Provides the weights and biases for the neural network.", file=file)
print("// Use generate_plicnet.py to generate this file for a given Pytorch model.\n", file=file)

print("#ifndef IRL_INTERFACE_RECONSTRUCTION_METHODS_PLICNET_HELPERS_H_", file=file)
print("#define IRL_INTERFACE_RECONSTRUCTION_METHODS_PLICNET_HELPERS_H_\n", file=file)

print("#include <array>\n", file=file)
print("namespace plicnet {\n", file=file)

print(f"static constexpr double tol = 1.0e-12;\n", file=file)    

param_info = []
count = 0

for param in model.parameters():
    count = count + 1
    name = ""
    if count%2 != 0:
        name = "lay" + str(int(count/2)+1) + "_weight"
        param_info.append({'name': name, 'type': 'weight', 'data': param.detach().numpy()})
    else:
        name = "lay" + str(int((count-1)/2)+1) + "_bias"
        param_info.append({'name': name, 'type': 'bias', 'data': param.detach().numpy()})

for info in param_info:
    name = info['name']
    data = info['data']
    if info['type'] == 'weight':
        out_features = data.shape[0]
        in_features = data.shape[1]
        print(f"static constexpr std::array<std::array<double, {in_features}>, {out_features}> {name} {{{{", file=file)    
        for m in range(out_features):
            row_data = data[m, :]
            values_str = np.array2string(row_data, separator=', ')[1:-1].strip()
            # Clean up newlines added by numpy to keep formatting precise
            values_str = values_str.replace('\n', '\n    ')
            formatted_values = format_array_content(values_str)
            comma = "," if m < out_features - 1 else ""
            print(f"    {{{{{formatted_values}}}}}{comma}", file=file)
        print("}};\n", file=file)
    elif info['type'] == 'bias':
        total_size = data.shape[0]
        print(f"static constexpr std::array<double, {total_size}> {name} {{{{", file=file)    
        values_str = np.array2string(data, separator=', ')[1:-1].strip()
        values_str = values_str.replace('\n', '\n    ')
        formatted_values = format_array_content(values_str)
        print(f"    {formatted_values}", file=file)
        print("}};\n", file=file)

print("} // namespace plicnet\n", file=file)
print("#endif // IRL_INTERFACE_RECONSTRUCTION_METHODS_PLICNET_HELPERS_H_", file=file)
file.close()